# RoadShield Vision — real RDD2022 training (Google Colab, free GPU if available)

This runs the repository's real-data pipeline, NOT the 48-image synthetic smoke test.
- First use Runtime → Change runtime type → GPU. **Free GPUs are not guaranteed**; stop if CUDA unavailable.
- Source is official Figshare article 21431547; the downloader verifies licence and MD5.
- Start with one country (e.g. India) to conserve bandwidth and free GPU time. Run a real held-out evaluation.
- Training saves **last.pt** at every epoch and **model.pt** for the best validation checkpoint. The notebook stores both in your Google Drive so you can continue with `--resume` after a Colab disconnect.
- Do not promote or claim model accuracy before a genuine real held-out test and separate Egypt-road validation.

**Until PR #1 is merged, clone its feature branch — not main.** Any GPU billing/upgrade requires separate approval.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU connected. Stop now to avoid wasting hours on CPU; select a free GPU runtime when available."
!nvidia-smi -L
!git clone --depth 1 --single-branch --branch claude/roadshield-ai-production-u7mtac https://github.com/Alaa2134/roadshield-ai.git
%cd roadshield-ai
# Preserve Colab's CUDA-enabled torch/torchvision. Install only optional export libraries.
!pip -q install onnx onnxscript
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/RoadShield/rs-frcnn-mbv3')
OUT.mkdir(parents=True, exist_ok=True)
print("Persistent checkpoints:", OUT)

## 1. Download (official source, licence-checked)

In [ ]:
!python ml/datasets/download_rdd2022.py --out /content/rdd2022 --dry-run
import json
record = json.load(open('/content/rdd2022/LICENSE_RECORD.json'))
assert record['allowed'], "Dataset licence needs review."
print("Official Figshare files:", [(x['name'], round((x['size'] or 0)/1e6)) for x in record['files']])

In [ ]:
# Start with one country archive, when present in the official Figshare record.
# Check the printed file list first; this match is not an invented URL or mirror.
FILE_MATCH = "India"
assert any(FILE_MATCH.lower() in f['name'].lower() for f in record['files']), "Choose a match shown in the official Figshare file list"
!python ml/datasets/download_rdd2022.py --out /content/rdd2022 --match {FILE_MATCH}
import zipfile
from pathlib import Path
DATA = Path('/content/rdd2022')
archives = list(DATA.glob('*.zip'))
assert archives, "No matching official ZIP downloaded; check FILE_MATCH."
for archive in archives:
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(DATA)
xml_dirs = [p for p in DATA.rglob('xmls') if p.is_dir() and p.parent.name == 'annotations' and p.parent.parent.name == 'train']
assert xml_dirs, "Cannot locate official VOC train/annotations/xmls structure; inspect ZIP layout."
RAW = str(xml_dirs[0].parents[3])
print("Detected dataset root:", RAW)

## 2. Prepare

In [ ]:
!python ml/datasets/prepare_rdd2022.py --raw {RAW} --out /content/rdd2022_prepared --seed 42
import json
manifest = json.load(open('/content/rdd2022_prepared/MANIFEST.json'))
print("Version:", manifest['version'], "Split counts:", manifest['counts'])
assert all(manifest['counts'].get(split, 0) > 0 for split in ('train','val','test')), "Cannot train on empty splits"

## 3–4. Train and evaluate

In [ ]:
import subprocess, sys
# Same config every resume (24 total epochs); conservative batch for a free T4.
cmd = [sys.executable, 'ml/detector/train_detector.py', '--data', '/content/rdd2022_prepared',
       '--raw', RAW, '--out', str(OUT), '--epochs', '24', '--batch', '2', '--seed', '42', '--pretrained', '--onnx']
if (OUT / 'last.pt').exists():
    cmd.append('--resume')
print("Resuming existing training" if '--resume' in cmd else "Starting real RDD2022 training")
subprocess.run(cmd, check=True)

In [ ]:
import json
metrics_path = OUT / 'metrics.json'
assert metrics_path.is_file(), "No held-out metrics yet; training has not finished."
m = json.loads(metrics_path.read_text())
assert not m.get('smoke_test'), "This is smoke-test data, NOT a real quality metric."
print('Dataset version:', m['dataset_version'], 'Train/val/test:', m['split_sizes'])
print(json.dumps(m['test'], indent=2))
print('By country:', json.dumps(m['per_country'], indent=2))
print('Latency:', m['latency_ms'])

## Artifacts / truthfulness

Google Drive retains `model.pt`, `last.pt`, `history.json`, `metrics.json` and (after the final epoch) `model.onnx`.
If Colab disconnects, run the notebook again with the same country and seed: `--resume` is added automatically if `last.pt` exists. The dataset may need downloading again, but verified archives are reused when present.
**Do not deploy the model automatically.** Assess labels/metrics, check the ONNX decoder in the AI service, and test on consented Egyptian road photos. Record real results in `docs/TRAINING_REPORT.md`.

In [ ]:
print('Persistent training directory:', OUT)
print([(p.name, round(p.stat().st_size/1e6, 1)) for p in sorted(OUT.iterdir()) if p.is_file()])
print('Only the metric files are suitable to download here; large weights remain in Drive.')